# Reader study — analysis frames

Four tidy frames built from `results/reader_study/{responses,source_ratings}.csv`.

| frame  | unit | rows | notes |
|--------|------|------|-------|
| `resp` | physician × question | 736 | raw columns + typed categoricals + derived `flow`, `support_level`, `helped`/`harmed`, `appropriate_reliance` |
| `cit`  | physician × question × citation | 1777 | source ratings joined to question metadata **and** rater attributes |
| `ques` | question | 272 | across-rater accuracy + citation support **pooled over all physicians** (Hypothesis A convention) |
| `part` | physician | 46 | accuracy, change/switch rates, `acc_gain` |

All four have a plain `RangeIndex`/single index, so `groupby` never hits index-vs-column
ambiguity. Ordered categoricals (`role`, `years_experience`, `derm_years`,
`disease_prevalence`, `support_level`, `flow`) so `groupby`/plots sort correctly and models
get sane reference levels.

Two export quirks handled here: `gp_years` is entirely empty (dropped), and `role` contains
`'rheumatologist'` lowercase (normalised to `Rheumatologist`).


In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80, "display.width", 200)

DATA_DIR = Path("..") / "results" / "reader_study"

# Ordered levels so groupby/plots sort correctly and models get sane reference categories.
EXPERIENCE_ORDER = ["< 1 year", "1-3 years", "4-9 years", "10+ years"]
PREVALENCE_ORDER = ["common", "moderate", "rare"]
ROLE_ORDER = ["Dermatology resident", "Dermatologist", "Rheumatologist"]

QUESTION_META = [
    "question_id", "case_index", "n_sources", "correct_choice", "correct_answer_text",
    "llm_answer", "llm_answer_text", "llm_is_correct", "question_type",
    "question_structure", "disease_category", "disease_subcategory",
    "disease_prevalence", "original_disease", "canonical_disease",
]


def _ordered(s, order):
    return pd.Categorical(s, categories=order, ordered=True)


def load_responses(data_dir=DATA_DIR):
    """One row per physician x question, with typed categoricals and derived flow labels."""
    df = pd.read_csv(Path(data_dir) / "responses.csv")

    # `gp_years` is empty in the current export; keep it only if it ever has data.
    if df["gp_years"].isna().all():
        df = df.drop(columns=["gp_years"])

    # 'rheumatologist' arrives lowercase in the export; normalise before ordering.
    df["role"] = _ordered(df["role"].str.strip().str.capitalize(), ROLE_ORDER)
    df["years_experience"] = _ordered(df["years_experience"], EXPERIENCE_ORDER)
    df["derm_years"] = _ordered(df["derm_years"], EXPERIENCE_ORDER)
    df["disease_prevalence"] = _ordered(df["disease_prevalence"], PREVALENCE_ORDER)
    for c in ["question_type", "question_structure", "disease_category",
              "disease_subcategory", "canonical_disease", "original_disease"]:
        df[c] = df[c].astype("category")
    df["group"] = df["group"].astype("category")

    # ── derived: citation support at the physician-question level ────────────
    df["support_frac"] = np.where(df["n_support_rated"] > 0,
                                  df["n_support"] / df["n_support_rated"], np.nan)
    df["support_level"] = pd.Categorical(
        np.select(
            [df["n_support_rated"].eq(0), df["n_support"].eq(0),
             df["support_frac"].lt(1.0), df["support_frac"].eq(1.0)],
            ["unrated", "none", "partial", "all"], default="unrated"),
        categories=["none", "partial", "all", "unrated"], ordered=True)

    # ── derived: reliance / change flow ─────────────────────────────────────
    df["flow"] = pd.Categorical(
        df["initial_correct"].map({True: "correct", False: "incorrect"}) + "->" +
        df["final_correct"].map({True: "correct", False: "incorrect"}),
        categories=["correct->correct", "correct->incorrect",
                    "incorrect->correct", "incorrect->incorrect"], ordered=True)
    df["helped"] = df["changed_answer"] & ~df["initial_correct"] & df["final_correct"]
    df["harmed"] = df["changed_answer"] & df["initial_correct"] & ~df["final_correct"]
    # Appropriate reliance: agree with the LLM when it is right, keep your own when it is wrong.
    df["appropriate_reliance"] = df["final_eq_llm"] == df["llm_is_correct"]

    return df.sort_values(["pid", "case_index"]).reset_index(drop=True)


def load_ratings(data_dir=DATA_DIR, responses=None):
    """One row per physician x question x citation, joined to question metadata."""
    df = pd.read_csv(Path(data_dir) / "source_ratings.csv")
    df["rating"] = pd.Categorical(df["rating"], ["Does not support", "Supports"], ordered=True)
    df["group"] = df["group"].astype("category")

    if responses is None:
        responses = load_responses(data_dir)
    meta = (responses[QUESTION_META].drop_duplicates("question_id")
            .drop(columns=["llm_answer", "llm_is_correct", "correct_choice", "case_index"]))
    df = df.merge(meta, on="question_id", how="left", validate="many_to_one")

    # Rater attributes, for rater-side analyses.
    who = responses[["pid", "role", "years_experience"]].drop_duplicates("pid")
    df = df.merge(who, on="pid", how="left", validate="many_to_one")

    return df.sort_values(["pid", "case_index", "source_position"]).reset_index(drop=True)


def load_questions(data_dir=DATA_DIR, responses=None, ratings=None):
    """
    Question level (unit = question, for Hypothesis A).

    Citation support is pooled across ALL physicians who rated that question:
    `any_supports` is True if any physician marked any citation as Supports.
    """
    if responses is None:
        responses = load_responses(data_dir)
    if ratings is None:
        ratings = load_ratings(data_dir, responses)

    q = (responses.groupby("question_id", observed=True)
         .agg(n_raters=("pid", "nunique"),
              initial_acc=("initial_correct", "mean"),
              final_acc=("final_correct", "mean"),
              change_rate=("changed_answer", "mean"),
              final_agree_llm=("final_eq_llm", "mean"),
              **{c: (c, "first") for c in QUESTION_META if c != "question_id"}))

    pooled = (ratings.groupby("question_id", observed=True)
              .agg(any_supports=("supports_llm", "any"),
                   all_support=("supports_llm", "all"),
                   support_rate=("supports_llm", "mean"),
                   n_citation_ratings=("supports_llm", "size")))

    # How often physicians disagree about the same citation.
    cite = (ratings.groupby(["question_id", "source_position"], observed=True)["supports_llm"]
            .mean().rename("cite_support_rate").reset_index())
    disagree = (cite.assign(contested=lambda d: d.cite_support_rate.between(0.001, 0.999))
                .groupby("question_id")["contested"].mean().rename("frac_contested_citations"))

    return q.join(pooled).join(disagree)


def load_participants(data_dir=DATA_DIR, responses=None):
    """Physician level: one row per pid."""
    if responses is None:
        responses = load_responses(data_dir)
    return (responses.groupby("pid", observed=True)
            .agg(role=("role", "first"),
                 years_experience=("years_experience", "first"),
                 group=("group", "first"),
                 n_questions=("question_id", "size"),
                 initial_acc=("initial_correct", "mean"),
                 final_acc=("final_correct", "mean"),
                 llm_acc=("llm_is_correct", "mean"),
                 change_rate=("changed_answer", "mean"),
                 switch_to_llm_rate=("switched_to_llm", "mean"),
                 switch_from_llm_rate=("switched_from_llm", "mean"),
                 final_agree_llm=("final_eq_llm", "mean"),
                 appropriate_reliance=("appropriate_reliance", "mean"),
                 n_helped=("helped", "sum"),
                 n_harmed=("harmed", "sum"),
                 mean_support_frac=("support_frac", "mean"))
            .assign(acc_gain=lambda d: d.final_acc - d.initial_acc))


@dataclass
class ReaderStudy:
    responses: pd.DataFrame
    ratings: pd.DataFrame
    questions: pd.DataFrame
    participants: pd.DataFrame


def load_reader_study(data_dir=DATA_DIR):
    responses = load_responses(data_dir)
    ratings = load_ratings(data_dir, responses)
    return ReaderStudy(
        responses=responses,
        ratings=ratings,
        questions=load_questions(data_dir, responses, ratings),
        participants=load_participants(data_dir, responses),
    )


ds = load_reader_study()

resp = ds.responses      # physician x question           (one row per answer)
cit = ds.ratings         # physician x question x citation (long, + question metadata)
ques = ds.questions      # question level                  (unit = question)
part = ds.participants   # physician level                 (one row per pid)

for name, df in [("resp", resp), ("cit", cit), ("ques", ques), ("part", part)]:
    print(f"{name:5s} {str(df.shape):10s} {df.shape[1]} cols")


resp  (736, 40)  40 cols
cit   (1777, 23) 23 cols
ques  (272, 24)  24 cols
part  (46, 16)   16 cols


In [2]:
resp.head(3)


,pid,role,derm_years,years_experience,group,case_index,question_id,n_sources,correct_choice,correct_answer_text,llm_answer,llm_answer_text,llm_is_correct,initial_answer,final_answer,initial_correct,final_correct,answered,changed_answer,initial_eq_llm,final_eq_llm,switched_to_llm,switched_from_llm,n_support,n_support_rated,all_sources_support,any_source_supports,question_type,disease_category,disease_prevalence,original_disease,canonical_disease,disease_subcategory,question_structure,support_frac,support_level,flow,helped,harmed,appropriate_reliance
0,p01,Dermatologist,10+ years,10+ years,1,1,1254,3,C,High TSH,C,High TSH,True,C,C,True,True,True,False,True,True,False,False,1,3,False,True,diagnosis,Metabolic Systemic,common,Hypothyroidism,Hypothyroidism,Endocrine-Associated,vignette,0.333333,partial,correct->correct,False,False,True
1,p01,Dermatologist,10+ years,10+ years,1,2,4915,1,B,located on buttock,B,located on buttock,True,C,D,False,False,True,True,False,False,False,False,1,1,True,True,diagnosis,Neoplastic,moderate,Dysplastic Nevus,Dysplastic Nevus,Premalignant,short_stem,1.000000,all,incorrect->incorrect,False,False,False
2,p01,Dermatologist,10+ years,10+ years,1,3,554,3,A,Transpeptidase,A,Transpeptidase,True,A,A,True,True,True,False,True,True,False,False,1,3,False,True,treatment,Infectious,moderate,Syphilis,Syphilis,Bacterial,vignette,0.333333,partial,correct->correct,False,False,True


In [3]:
cit.head(3)


,pid,group,case_index,question_id,source_position,document_id,rating,supports_llm,llm_answer,correct_choice,llm_is_correct,n_sources,correct_answer_text,llm_answer_text,question_type,question_structure,disease_category,disease_subcategory,disease_prevalence,original_disease,canonical_disease,role,years_experience
0,p01,1,1,1254,1,1,Does not support,False,C,C,True,3,High TSH,High TSH,diagnosis,vignette,Metabolic Systemic,Endocrine-Associated,common,Hypothyroidism,Hypothyroidism,Dermatologist,10+ years
1,p01,1,1,1254,2,2,Does not support,False,C,C,True,3,High TSH,High TSH,diagnosis,vignette,Metabolic Systemic,Endocrine-Associated,common,Hypothyroidism,Hypothyroidism,Dermatologist,10+ years
2,p01,1,1,1254,3,7,Supports,True,C,C,True,3,High TSH,High TSH,diagnosis,vignette,Metabolic Systemic,Endocrine-Associated,common,Hypothyroidism,Hypothyroidism,Dermatologist,10+ years


In [4]:
display(ques.head(3))
part.head(3)


,n_raters,initial_acc,final_acc,change_rate,final_agree_llm,case_index,n_sources,correct_choice,correct_answer_text,llm_answer,llm_answer_text,llm_is_correct,question_type,question_structure,disease_category,disease_subcategory,disease_prevalence,original_disease,canonical_disease,any_supports,all_support,support_rate,n_citation_ratings,frac_contested_citations
question_id,,,,,,,,,,,,,,,,,,,,,,,,
10,3,1.000000,1.000000,0.0,1.000000,1,2,A,Squamous cell carcinoma,A,Squamous cell carcinoma,True,diagnosis,vignette,Neoplastic,Malignant,common,Squamous Cell Carcinoma,Squamous Cell Carcinoma,True,False,0.333333,6,0.5
54,3,0.666667,0.666667,0.0,0.666667,8,2,C,High TSH,C,High TSH,True,diagnosis,vignette,Metabolic Systemic,Endocrine-Associated,common,Hypothyroidism,Hypothyroidism,True,False,0.500000,6,1.0
57,3,1.000000,1.000000,0.0,1.000000,6,3,B,Scalded skin syndrome,B,Scalded skin syndrome,True,diagnosis,vignette,Infectious,Bacterial,moderate,Staphylococcal Scalded Skin Syndrome,Staphylococcal Scalded Skin Syndrome,True,False,0.333333,9,0.0


,role,years_experience,group,n_questions,initial_acc,final_acc,llm_acc,change_rate,switch_to_llm_rate,switch_from_llm_rate,final_agree_llm,appropriate_reliance,n_helped,n_harmed,mean_support_frac,acc_gain
pid,,,,,,,,,,,,,,,,
p01,Dermatologist,10+ years,1,16,0.6875,0.6875,0.875,0.1875,0.0625,0.0625,0.6875,0.6875,1,1,0.651042,0.00
p02,Dermatologist,4-9 years,1,16,0.4375,0.6875,0.875,0.3125,0.2500,0.0000,0.8125,0.6875,4,0,0.697917,0.25
p03,Dermatologist,4-9 years,1,16,0.9375,0.9375,0.875,0.1250,0.1250,0.0000,0.9375,0.9375,1,1,0.932292,0.00


## Helper: grouped rates with Wilson CIs

`rate(df, "final_correct", by="role")` → one row per group with `n`, `k`, `rate`, `lo`, `hi`.
Works on any of the four frames.


In [5]:
def wilson(k, n, z=1.96):
    """Wilson score interval — behaves at rates near 0/1 where the normal approx does not."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = p + z**2 / (2 * n)
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((c - h) / d, (c + h) / d)


def rate(df, col, by=None):
    """Proportion of a boolean column, optionally grouped. Returns n, k, rate, lo, hi."""
    def _one(g):
        s = g[col].dropna().astype(bool)
        lo, hi = wilson(s.sum(), len(s))
        return pd.Series({"n": len(s), "k": int(s.sum()), "rate": s.mean(), "lo": lo, "hi": hi})

    if by is None:
        return _one(df).to_frame(col).T
    return (df.groupby(by, observed=True)[[col]].apply(_one)
            .assign(n=lambda d: d.n.astype(int), k=lambda d: d.k.astype(int)))


rate(resp, "final_correct", by="role")


,n,k,rate,lo,hi
role,,,,,
Dermatology resident,32,26,0.812500,0.646905,0.911106
Dermatologist,688,568,0.825581,0.795439,0.852108
Rheumatologist,16,11,0.687500,0.444039,0.858356


## Accuracy before vs. after seeing the LLM answer


In [6]:
overall = pd.concat({
    "initial": rate(resp, "initial_correct").iloc[0],
    "final": rate(resp, "final_correct").iloc[0],
    "llm": rate(resp, "llm_is_correct").iloc[0],
}, axis=1).T
display(overall)

# Physician-level paired change (the unit that respects clustering by rater)
display(part[["initial_acc", "final_acc", "acc_gain", "n_helped", "n_harmed"]].describe().round(3))

resp.groupby("role", observed=True)[
    ["initial_correct", "final_correct", "llm_is_correct", "changed_answer",
     "switched_to_llm", "switched_from_llm", "appropriate_reliance"]
].mean().round(3)


,n,k,rate,lo,hi
initial,736.0,515.0,0.699728,0.665645,0.731737
final,736.0,605.0,0.822011,0.792725,0.847952
llm,736.0,644.0,0.875000,0.849142,0.896963


,initial_acc,final_acc,acc_gain,n_helped,n_harmed
count,46.000,46.000,46.000,46.000,46.000
mean,0.700,0.822,0.122,2.478,0.522
std,0.141,0.091,0.101,1.574,0.658
min,0.375,0.625,-0.125,0.000,0.000
25%,0.625,0.750,0.062,1.250,0.000
50%,0.688,0.812,0.125,2.000,0.000
75%,0.750,0.875,0.188,3.750,1.000
max,1.000,1.000,0.312,6.000,2.000


,initial_correct,final_correct,llm_is_correct,changed_answer,switched_to_llm,switched_from_llm,appropriate_reliance
role,,,,,,,
Dermatology resident,0.750,0.812,0.875,0.094,0.094,0.00,0.844
Dermatologist,0.705,0.826,0.875,0.208,0.183,0.01,0.852
Rheumatologist,0.375,0.688,0.875,0.562,0.438,0.00,0.688


## Answer-change flow and reliance

`flow` crosses initial × final correctness; splitting it by whether the LLM was right shows
where the help and the harm come from.


In [7]:
flow_tab = pd.crosstab(resp["flow"], resp["llm_is_correct"], margins=True)
display(flow_tab)

# Row-normalised: given the LLM was right/wrong, where did physicians end up?
display(pd.crosstab(resp["llm_is_correct"], resp["flow"], normalize="index").round(3))

# Who changes their mind, and does it pay off?
display(resp.groupby(["llm_is_correct", "initial_eq_llm"], observed=True)
        .agg(n=("changed_answer", "size"),
             change_rate=("changed_answer", "mean"),
             final_acc=("final_correct", "mean")).round(3))


llm_is_correct,False,True,All
flow,,,
correct->correct,30,461,491
correct->incorrect,18,6,24
incorrect->correct,0,114,114
incorrect->incorrect,44,63,107
All,92,644,736


flow,correct->correct,correct->incorrect,incorrect->correct,incorrect->incorrect
llm_is_correct,,,,
False,0.326,0.196,0.000,0.478
True,0.716,0.009,0.177,0.098


n  change_rate  final_acc
llm_is_correct initial_eq_llm                             
False          False            70        0.329      0.429
               True             22        0.045      0.000
True           False           177        0.706      0.644
               True            467        0.013      0.987

## Citation support

Two units, deliberately kept separate:
- **`resp.support_level` / `support_frac`** — physician-question level (each physician's own ratings) → Hypotheses B/C.
- **`ques.support_rate` / `any_supports`** — pooled across all physicians who rated the question → Hypothesis A.


In [8]:
# Physician-question level: does perceived support track ending up correct?
display(rate(resp, "final_correct", by="support_level"))
display(rate(resp, "final_eq_llm", by="support_level"))

# Question level: support vs. whether the LLM was actually right
display(ques.groupby("llm_is_correct")[
    ["support_rate", "any_supports", "all_support", "frac_contested_citations",
     "initial_acc", "final_acc"]].mean().round(3))

# Citation level: raters' support rate by position in the source list
display(cit.groupby("source_position")["supports_llm"].agg(["mean", "size"]).round(3))


,n,k,rate,lo,hi
support_level,,,,,
none,168,112,0.666667,0.592359,0.733522
partial,272,233,0.856618,0.809995,0.893307
all,296,260,0.878378,0.836218,0.910843


,n,k,rate,lo,hi
support_level,,,,,
none,168,96,0.571429,0.495823,0.643841
partial,272,247,0.908088,0.867841,0.936969
all,296,275,0.929054,0.893982,0.953132


,support_rate,any_supports,all_support,frac_contested_citations,initial_acc,final_acc
llm_is_correct,,,,,,
False,0.391,0.824,0.088,0.534,0.520,0.328
True,0.626,0.903,0.223,0.396,0.722,0.891


,mean,size
source_position,,
1,0.637,736
2,0.604,584
3,0.557,361
4,0.701,87
5,0.889,9


## Subgroup cuts

Same recipe for any question attribute — swap the `by=` column.


In [9]:
def subgroup(by, df=resp):
    """initial/final accuracy, LLM accuracy and change rate for each level of `by`."""
    return (df.groupby(by, observed=True)
            .agg(n=("final_correct", "size"),
                 initial_acc=("initial_correct", "mean"),
                 final_acc=("final_correct", "mean"),
                 llm_acc=("llm_is_correct", "mean"),
                 change_rate=("changed_answer", "mean"),
                 support_frac=("support_frac", "mean"))
            .assign(acc_gain=lambda d: d.final_acc - d.initial_acc)
            .round(3))


for col in ["question_type", "question_structure", "disease_prevalence",
            "disease_category", "n_sources"]:
    print(f"\n── by {col} " + "─" * 40)
    display(subgroup(col))



── by question_type ────────────────────────────────────────


,n,initial_acc,final_acc,llm_acc,change_rate,support_frac,acc_gain
question_type,,,,,,,
diagnosis,364,0.745,0.824,0.865,0.201,0.645,0.080
epidemiology,41,0.659,0.805,0.854,0.171,0.563,0.146
mechanism,132,0.606,0.765,0.902,0.250,0.527,0.159
prognosis,46,0.522,0.848,0.804,0.370,0.561,0.326
treatment,153,0.739,0.863,0.902,0.163,0.582,0.124



── by question_structure ────────────────────────────────────────


,n,initial_acc,final_acc,llm_acc,change_rate,support_frac,acc_gain
question_structure,,,,,,,
short_stem,251,0.757,0.841,0.888,0.171,0.676,0.084
vignette,485,0.670,0.812,0.868,0.231,0.562,0.142



── by disease_prevalence ────────────────────────────────────────


,n,initial_acc,final_acc,llm_acc,change_rate,support_frac,acc_gain
disease_prevalence,,,,,,,
common,241,0.797,0.867,0.946,0.108,0.584,0.071
moderate,299,0.679,0.796,0.823,0.214,0.587,0.117
rare,196,0.612,0.806,0.867,0.332,0.644,0.194



── by disease_category ────────────────────────────────────────


,n,initial_acc,final_acc,llm_acc,change_rate,support_frac,acc_gain
disease_category,,,,,,,
Genetic Rare,104,0.702,0.856,0.856,0.308,0.684,0.154
Hair Nail,9,0.778,1.000,1.000,0.222,0.639,0.222
Infectious,176,0.648,0.812,0.875,0.182,0.572,0.165
Inflammatory,164,0.726,0.823,0.878,0.159,0.643,0.098
Metabolic Systemic,93,0.677,0.871,0.935,0.247,0.591,0.194
Neoplastic,130,0.700,0.723,0.823,0.231,0.474,0.023
Other,30,0.900,0.967,1.000,0.067,0.711,0.067
Pigmentation,30,0.700,0.833,0.800,0.267,0.708,0.133



── by n_sources ────────────────────────────────────────


,n,initial_acc,final_acc,llm_acc,change_rate,support_frac,acc_gain
n_sources,,,,,,,
1,152,0.651,0.691,0.750,0.197,0.533,0.039
2,223,0.673,0.843,0.883,0.251,0.619,0.170
3,274,0.701,0.861,0.931,0.201,0.601,0.161
4,78,0.859,0.885,0.923,0.154,0.670,0.026
5,9,0.778,0.778,0.667,0.222,0.711,0.000


## Per-group sanity check

Does every question set (`group`) contribute comparably, or do one or two swing the headline?
Three views: (1) per-group accuracy with CIs, (2) leave-one-group-out recomputation of the
headline numbers, (3) is the between-group spread bigger than sampling noise.


In [10]:
by_group = (resp.groupby("group", observed=True)
            .agg(n_phys=("pid", "nunique"),
                 n_questions=("question_id", "nunique"),
                 n=("final_correct", "size"),
                 initial_acc=("initial_correct", "mean"),
                 final_acc=("final_correct", "mean"),
                 llm_acc=("llm_is_correct", "mean"),
                 change_rate=("changed_answer", "mean"),
                 support_frac=("support_frac", "mean"))
            .assign(acc_gain=lambda d: d.final_acc - d.initial_acc))

# Wilson CI on each group's final accuracy, to see which gaps are real vs. n=32-48 noise.
by_group = by_group.join(rate(resp, "final_correct", by="group")[["lo", "hi"]])
display(by_group.round(3))

print(f"physicians per group: {by_group.n_phys.value_counts().to_dict()}"
      f"   (total {by_group.n_phys.sum()})")
print(f"questions per group : {sorted(by_group.n_questions.unique())}")
print(f"LLM accuracy per group: {sorted(resp.groupby('group', observed=True).llm_is_correct.mean().unique().round(3))}"
      "  <- fixed by design, so groups are not confounded by an easier/harder LLM")


,n_phys,n_questions,n,initial_acc,final_acc,llm_acc,change_rate,support_frac,acc_gain,lo,hi
group,,,,,,,,,,,
1,3,16,48,0.688,0.771,0.875,0.208,0.760,0.083,0.635,0.867
2,3,16,48,0.667,0.812,0.875,0.229,0.818,0.146,0.681,0.898
3,3,16,48,0.792,0.833,0.875,0.062,0.517,0.042,0.704,0.913
4,2,16,32,0.594,0.812,0.875,0.406,0.672,0.219,0.647,0.911
5,3,16,48,0.604,0.750,0.875,0.229,0.542,0.146,0.612,0.851
6,3,16,48,0.729,0.854,0.875,0.208,0.583,0.125,0.728,0.928
7,3,16,48,0.708,0.854,0.875,0.271,0.467,0.146,0.728,0.928
8,2,16,32,0.719,0.844,0.875,0.188,0.607,0.125,0.682,0.931
9,3,16,48,0.688,0.792,0.875,0.146,0.580,0.104,0.657,0.883


physicians per group: {3: 13, 2: 3, 1: 1}   (total 46)
questions per group : [16]
LLM accuracy per group: [0.875]  <- fixed by design, so groups are not confounded by an easier/harder LLM


In [11]:
# Leave-one-group-out: recompute the headline numbers 17 times, each time dropping one group.
# If a single group were driving the result, its row would sit far from the rest.
loo = pd.DataFrame([
    dict(dropped=g,
         n=len(sub),
         initial_acc=sub.initial_correct.mean(),
         final_acc=sub.final_correct.mean(),
         acc_gain=sub.final_correct.mean() - sub.initial_correct.mean(),
         appropriate_reliance=sub.appropriate_reliance.mean())
    for g in resp["group"].cat.categories
    for sub in [resp[resp["group"] != g]]
]).set_index("dropped")

full = dict(final_acc=resp.final_correct.mean(),
            acc_gain=resp.final_correct.mean() - resp.initial_correct.mean())
display(loo.round(4))
print(f"full sample : final_acc={full['final_acc']:.4f}  acc_gain={full['acc_gain']:.4f}")
print(f"LOO final_acc range: {loo.final_acc.min():.4f} – {loo.final_acc.max():.4f} "
      f"(max shift {abs(loo.final_acc - full['final_acc']).max():.4f})")
print(f"LOO acc_gain  range: {loo.acc_gain.min():.4f} – {loo.acc_gain.max():.4f} "
      f"(max shift {abs(loo.acc_gain - full['acc_gain']).max():.4f})")


,n,initial_acc,final_acc,acc_gain,appropriate_reliance
dropped,,,,,
1,688,0.7006,0.8256,0.1250,0.8532
2,688,0.7020,0.8227,0.1206,0.8488
3,688,0.6933,0.8212,0.1279,0.8488
4,704,0.7045,0.8224,0.1179,0.8494
5,688,0.7064,0.8270,0.1206,0.8532
6,688,0.6977,0.8198,0.1221,0.8459
7,688,0.6991,0.8198,0.1206,0.8445
8,704,0.6989,0.8210,0.1222,0.8480
9,688,0.7006,0.8241,0.1235,0.8459


full sample : final_acc=0.8220  acc_gain=0.1223
LOO final_acc range: 0.8183 – 0.8270 (max shift 0.0050)
LOO acc_gain  range: 0.1105 – 0.1294 (max shift 0.0118)


In [12]:
from scipy import stats

# Is the between-group spread larger than what sampling noise alone would produce?
p = resp.final_correct.mean()
obs_sd = by_group.final_acc.std()
exp_sd = np.sqrt(p * (1 - p) / by_group.n).mean()
print(f"between-group SD of final_acc: observed {obs_sd:.4f} vs {exp_sd:.4f} expected from "
      f"binomial noise alone  (ratio {obs_sd / exp_sd:.2f})")

# Homogeneity tests. The chi2 ignores clustering within physician, so treat it as a rough
# screen; the Kruskal-Wallis on physician-level gain uses one observation per physician.
for col in ["initial_correct", "final_correct"]:
    chi2, pv, dof, _ = stats.chi2_contingency(pd.crosstab(resp["group"], resp[col]))
    print(f"chi2 homogeneity of {col:16s} across groups: chi2={chi2:5.2f}  dof={dof}  p={pv:.3f}")

phys_gain = (resp.groupby(["group", "pid"], observed=True)[["initial_correct", "final_correct"]]
             .mean().assign(gain=lambda d: d.final_correct - d.initial_correct))
kw = stats.kruskal(*[v.values for _, v in phys_gain.groupby(level="group", observed=True)["gain"]])
print(f"Kruskal-Wallis on physician-level acc_gain by group: H={kw.statistic:.2f}  p={kw.pvalue:.3f}")
print(f"physician-level gain: mean {phys_gain.gain.mean():.4f}, sd {phys_gain.gain.std():.4f}")


between-group SD of final_acc: observed 0.0392 vs 0.0598 expected from binomial noise alone  (ratio 0.66)
chi2 homogeneity of initial_correct  across groups: chi2=17.37  dof=16  p=0.362
chi2 homogeneity of final_correct    across groups: chi2= 6.59  dof=16  p=0.980
Kruskal-Wallis on physician-level acc_gain by group: H=17.30  p=0.366
physician-level gain: mean 0.1223, sd 0.1012


In [13]:
# scratch
